In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sys
sys.path.append("../src")
from model import (load_features, prepare_model_data, time_series_cv,
                   train_final_model, compute_shap, explain_single_prediction)

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
df = load_features(filepath="../data/processed/features.csv")
X, y = prepare_model_data(df)
print(f"\nFeature columns:\n{list(X.columns)}")

In [ ]:
cv_results = time_series_cv(X, y, n_splits=5)
print(f"\nMAPE below 10% = model is production-grade")
print(f"Your MAPE: {cv_results['mean_mape']}%")

In [ ]:
model, metrics, X_test, y_test, preds = train_final_model(X, y)
print(f"\nR² score: {metrics['test_r2']} (closer to 1.0 = better)")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test[:300], preds[:300], alpha=0.4, color="steelblue", s=15)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], "r--", linewidth=1)
plt.xlabel("Actual Demand")
plt.ylabel("Predicted Demand")
plt.title("Actual vs Predicted (Test Set)")

plt.subplot(1, 2, 2)
residuals = y_test.values - preds
plt.hist(residuals, bins=50, color="coral", edgecolor="white")
plt.axvline(0, color="black", linestyle="--")
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Count")
plt.title("Residual Distribution")

plt.tight_layout()
plt.show()

In [ ]:
import shap

sample_X    = X.sample(500, random_state=42)
explainer, shap_values = compute_shap(model, sample_X, save_dir="../models")

shap.summary_plot(shap_values, sample_X, plot_type="bar", max_display=15)

In [ ]:
shap.summary_plot(shap_values, sample_X, max_display=15)

In [ ]:
example = explain_single_prediction(explainer, sample_X, row_idx=0)

print("Why did the model predict this demand level?\n")
for r in example["top_3_reasons"]:
    arrow = "▲" if r["direction"] == "increased" else "▼"
    print(f"  {arrow}  {r['feature']:35s} = {r['feature_value']:.3f}"
          f"  →  demand {r['direction']} by {abs(r['shap_value']):.3f}")

In [ ]:
# Shows how base_price affects demand prediction
shap.dependence_plot("base_price", shap_values.values,
                     sample_X, interaction_index="inventory_ratio")
plt.title("SHAP Dependence: Price effect on Demand\n(colored by inventory ratio)")
plt.show()

In [ ]:
import json, os, joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/xgb_demand_model.pkl")

metrics["feature_names"] = X.columns.tolist()
with open("../models/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Model saved  →  ../models/xgb_demand_model.pkl")
print("Metrics saved →  ../models/model_metrics.json")
print(f"\nFinal metrics:")
for k, v in metrics.items():
    if k != "feature_names":
        print(f"  {k}: {v}")